### [Hybrid Attention for Binary Sequence Forecasting](https://pub.towardsai.net/hybrid-attention-for-binary-sequence-forecasting-637abff3f87d)

> Combining n-Gram Embeddings, Count-Aware Self-Attention, and Recency-Weighted ARMA for Multi-Horizon Distributional Predictions

#### Decoding Binary Sequence Dynamics

A binary time series {x_t}​ is a series of zeros and ones — machine faults, user clicks, daily rainfall indicators — where each new value depends on hidden patterns in its past.

##### Pipeline

1. Encode history as symbolic n-grams and statistical recency features.
2. Fuse them with self-attention to capture exact and fuzzy motifs.
3. Predict jointly the next bit and K-step count via dual heads.
4. Optimize a combined Bernoulli + Categorical loss under time-series CV.
5. Validate with point and distributional scores and check interval coverage.

In [1]:
!pip install -q numpy pandas matplotlib scikit-learn torch

In [2]:
import warnings
warnings.filterwarnings("ignore")

import math
import numpy as np
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

In [3]:
def ranked_probability_score(true_counts, cdf_preds):
    K = cdf_preds.shape[1] - 1
    return np.mean(np.sum((cdf_preds - (np.arange(K+1)[None, :] >= true_counts[:, None]))**2, axis=1))

In [4]:
class BinarySeriesDataset(Dataset):
    def __init__(self, series, K, ngram_vocab, recency_windows):
        self.series = series
        self.K = K
        self.vocab = ngram_vocab
        self.wins = recency_windows
        self.ngram_sizes = sorted({len(p) for p in ngram_vocab})
        self.offset = max(self.K, max(self.ngram_sizes), max(self.wins))
        self.length = len(series) - self.offset

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        t = idx + self.offset
        future = self.series[t:t+self.K]
        y_point = float(self.series[t])
        y_count = int(future.sum())
        patterns = []
        for n in self.ngram_sizes:
            pat = tuple(int(x) for x in self.series[t-n:t])
            patterns.append(self.vocab.get(pat, 0))
        patterns = np.array(patterns, dtype=np.int64)
        rec_feats = []
        for w in self.wins:
            window = self.series[t-w:t]
            rec_feats.append(window.mean())
            rec_feats.append((window[1:] != window[:-1]).mean() if w>1 else 0.0)
        rec_feats = np.array(rec_feats, dtype=np.float32)
        return patterns, rec_feats, y_point, y_count

In [5]:
class BinaryTrendFormer(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, recency_dim, hidden_dim, K, dropout_p=0.2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.rec_fc = nn.Sequential(nn.Linear(recency_dim, hidden_dim), nn.ReLU())
        self.fuse_fc = nn.Sequential(nn.Linear(embed_dim+hidden_dim, hidden_dim), nn.ReLU())
        self.dropout = nn.Dropout(dropout_p)
        self.point_head = nn.Linear(hidden_dim, 1)
        self.count_head = nn.Linear(hidden_dim, K+1)

    def forward(self, patterns, rec_feats):
        x = self.embed(patterns)
        attn_out, _ = self.attn(x, x, x)
        pat_feat = attn_out.mean(dim=1)
        rec_feat = self.rec_fc(rec_feats)
        fusion = torch.cat([pat_feat, rec_feat], dim=1)
        h = self.fuse_fc(fusion)
        h = self.dropout(h)
        p_logit = self.point_head(h).squeeze(-1)
        p_point = torch.sigmoid(p_logit)
        count_logits = self.count_head(h)
        return p_point, p_logit, count_logits

In [6]:
def train_and_evaluate(series, K=5,
                       ngram_range=(3,7),
                       recency_windows=[3,7,14],
                       embed_dim=64,
                       num_heads=4,
                       hidden_dim=128,
                       lr=1e-3,
                       weight_decay=1e-5,
                       batch_size=128,
                       epochs=20,
                       patience=5,
                       tscv_splits=3,
                       device='cpu'):

    # Prepare vocabulary
    min_n, max_n = ngram_range
    patterns = [tuple(series[t-n:t]) for t in range(max_n, len(series)) for n in range(min_n, max_n+1)]
    vocab = {pat: i+1 for i, pat in enumerate(set(patterns))}
    recency_dim = 2 * len(recency_windows)
    vocab_size = len(vocab) + 1

    tscv = TimeSeriesSplit(n_splits=tscv_splits)
    metrics = {'logloss':[], 'brier':[], 'auroc':[], 'rps':[], 'coverage':[]}
    last_model = None

    for train_idx, val_idx in tscv.split(series):
        tr, vl = series[train_idx], series[val_idx]
        tr_ds = BinarySeriesDataset(tr, K, vocab, recency_windows)
        vl_ds = BinarySeriesDataset(vl, K, vocab, recency_windows)
        tr_loader = DataLoader(tr_ds, batch_size=batch_size, shuffle=True)
        vl_loader = DataLoader(vl_ds, batch_size=batch_size)

        model = BinaryTrendFormer(vocab_size, embed_dim, num_heads,
                                  recency_dim, hidden_dim, K).to(device)
        opt = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(opt, 'min', patience=3, factor=0.5)
        bce = nn.BCEWithLogitsLoss()
        ce  = nn.CrossEntropyLoss()

        # Early-stop training
        best_val_loss = float('inf')
        wait = 0
        best_state = None

        for epoch in range(epochs):
            model.train()
            for p_np, r_np, y_pt, y_ct in tr_loader:
                p = torch.tensor(p_np, dtype=torch.long, device=device)
                r = torch.tensor(r_np, dtype=torch.float32, device=device)
                yp = torch.tensor(y_pt, dtype=torch.float32, device=device)
                yc = torch.tensor(y_ct, dtype=torch.long, device=device)
                opt.zero_grad()
                _, logit, cnt_logits = model(p, r)
                loss = bce(logit, yp) + ce(cnt_logits, yc)
                loss.backward()
                opt.step()

            # Validation loss (batch-wise)
            model.eval()
            val_losses = []
            with torch.no_grad():
                for p_np, r_np, y_pt, _ in vl_loader:
                    p = torch.tensor(p_np, dtype=torch.long, device=device)
                    r = torch.tensor(r_np, dtype=torch.float32, device=device)
                    yp = torch.tensor(y_pt, dtype=torch.float32, device=device)
                    _, logit, _ = model(p, r)
                    probs = torch.sigmoid(logit).cpu().numpy()
                    val_losses.append(log_loss(yp.cpu().numpy(), probs))
            val_loss = np.mean(val_losses)
            scheduler.step(val_loss)

            if val_loss < best_val_loss - 1e-4:
                best_val_loss = val_loss
                wait = 0
                best_state = model.state_dict()
            else:
                wait += 1
                if wait >= patience:
                    break

        model.load_state_dict(best_state)
        last_model = model

        # Compute fold metrics
        model.eval()
        all_p, all_cdf, all_counts = [], [], []
        with torch.no_grad():
            for p_np, r_np, _, y_ct in vl_loader:
                p = torch.tensor(p_np, dtype=torch.long, device=device)
                r = torch.tensor(r_np, dtype=torch.float32, device=device)
                p_pt, _, cnt_logits = model(p, r)
                all_p.extend(p_pt.cpu().numpy())
                probs = torch.softmax(cnt_logits, dim=1).cpu().numpy()
                all_cdf.extend(np.cumsum(probs, axis=1))
                all_counts.extend(y_ct)

        all_p      = np.array(all_p)
        all_cdf    = np.array(all_cdf)
        all_counts = np.array(all_counts)

        ll  = log_loss((all_counts>0).astype(int), all_p)
        bri = brier_score_loss((all_counts>0).astype(int), all_p)
        auc = roc_auc_score((all_counts>0).astype(int), all_p)
        rps = ranked_probability_score(all_counts, all_cdf)

        mean_cdf = all_cdf.mean(axis=0)
        low_idx = np.searchsorted(mean_cdf, 0.025, side='left')
        high_idx = np.searchsorted(mean_cdf, 0.975, side='left')
        cdf_low = all_cdf[:, low_idx]
        cdf_high = all_cdf[:, high_idx]
        cov = np.mean((all_counts>=cdf_low) & (all_counts<=cdf_high))

        metrics['logloss'].append(ll)
        metrics['brier'].append(bri)
        metrics['auroc'].append(auc)
        metrics['rps'].append(rps)
        metrics['coverage'].append(cov)

    # Validation summary
    for k in metrics:
        metrics[k] = np.mean(metrics[k])
    print("\n=== Validation Summary ===")
    print(f"1-step → LogLoss {metrics['logloss']:.3f}, Brier {metrics['brier']:.3f}, AUC {metrics['auroc']:.3f}")
    print(f"K-step → RPS {metrics['rps']:.3f}, 95% CI Cov {metrics['coverage']:.3f}\n")

    # Retrain on full series
    final_model = BinaryTrendFormer(vocab_size, embed_dim, num_heads,
                                    recency_dim, hidden_dim, K).to(device)
    opt = optim.Adam(final_model.parameters(), lr=lr, weight_decay=weight_decay)
    bce = nn.BCEWithLogitsLoss()
    ce  = nn.CrossEntropyLoss()
    full_loader = DataLoader(BinarySeriesDataset(series, K, vocab, recency_windows),
                              batch_size=batch_size, shuffle=True)
    final_model.train()
    for _ in range(epochs):
        for p_np, r_np, y_pt, y_ct in full_loader:
            p = torch.tensor(p_np, dtype=torch.long, device=device)
            r = torch.tensor(r_np, dtype=torch.float32, device=device)
            yp = torch.tensor(y_pt, dtype=torch.float32, device=device)
            yc = torch.tensor(y_ct, dtype=torch.long, device=device)
            opt.zero_grad()
            _, logit, cnt_logits = final_model(p, r)
            (bce(logit, yp) + ce(cnt_logits, yc)).backward()
            opt.step()

    # Final forecast and intervals
    final_model.eval()
    with torch.no_grad():
        ds = BinarySeriesDataset(series, K, vocab, recency_windows)
        p_np, r_np, _, _ = ds[-1]
        p = torch.tensor(p_np, dtype=torch.long, device=device).unsqueeze(0)
        r = torch.tensor(r_np, dtype=torch.float32, device=device).unsqueeze(0)
        p_pt, _, cnt_logits = final_model(p, r)
        p_next = float(p_pt.cpu())
        dist = torch.softmax(cnt_logits, dim=1).cpu().numpy().flatten()
        exp_count = np.dot(np.arange(K+1), dist)

        cdf = np.cumsum(dist)
        idx_low = np.searchsorted(cdf, 0.025, side='left')
        idx_high = np.searchsorted(cdf, 0.975, side='left')
        idx_low = min(max(idx_low, 0), K)
        idx_high = min(max(idx_high, 0), K)
        model_cl_low = cdf[idx_low]
        model_cl_hi  = cdf[idx_high]

        # Bernoulli-CLT intervals
        p_hat = series.mean()
        n_hat = len(series)
        se_p = math.sqrt(p_hat*(1-p_hat)/n_hat)
        clt_low = max(0, p_hat - 1.96*se_p)
        clt_hi  = min(1, p_hat + 1.96*se_p)
        mu_c = K * p_hat
        se_c = math.sqrt(K * p_hat * (1-p_hat))
        count_low = max(0, mu_c - 1.96*se_c)
        count_hi  = min(K, mu_c + 1.96*se_c)

    print(f"Final Next-step P(1): {p_next:.3f}")
    print(f" Model CDF 95% CI for P(1): [{model_cl_low:.3f}, {model_cl_hi:.3f}]")
    print(f" Bernoulli-CLT 95% CI for p: [{clt_low:.3f}, {clt_hi:.3f}] (n={n_hat})")
    print(f"Final Expected #1s in next {K}: {exp_count:.2f}")
    print(f" Model CDF 95% CI for count: [{model_cl_low*K:.2f}, {model_cl_hi*K:.2f}]")
    print(f" Bernoulli-CLT 95% CI for count: [{count_low:.2f}, {count_hi:.2f}]\n")

In [7]:
data_url = 'https://raw.githubusercontent.com/datalev001/purebin_tsm/refs/heads/main/data/bin_series.csv'
df_loaded = pd.read_csv(data_url)
series = df_loaded['value'].values
train_and_evaluate(series, K=5, epochs=20, tscv_splits=3, device='cpu')


=== Validation Summary ===
1-step → LogLoss 0.692, Brier 0.205, AUC 0.708
K-step → RPS 0.591, 95% CI Cov 0.073

Final Next-step P(1): nan
 Model CDF 95% CI for P(1): [nan, nan]
 Bernoulli-CLT 95% CI for p: [0.184, 0.220] (n=2000)
Final Expected #1s in next 5: nan
 Model CDF 95% CI for count: [nan, nan]
 Bernoulli-CLT 95% CI for count: [0.00, 2.77]



#### Summarized Overview

1. Feature engineering

* n-gram patterns — for every step `t` the last `n ∈ [min_n,max_n]` symbols are treated as a token; the set of all distinct tokens becomes a vocabulary that is embedded later.

* Recency statistics — inside a list of fixed windows (3, 7, 14) it computes the recent mean (local prevalence of “1”) and the recent switching-rate (local volatility).

2. Dataset object `BinarySeriesDataset` returns for each time index

- pattern IDs (patterns)
- recency features (rec_feats)
- the next symbol y₁ (point target)
- the sum of the next K symbols (count target).

3. Model — Binary TrendFormer

- Embedding layer maps each pattern ID to an E-dim vector.
- Single self-attention layer (nn.MultiheadAttention) lets each n-gram attend to the others, producing a pattern feature via mean-pooling.
- Recency MLP compresses the handcrafted statistics.
- Fusion MLP concatenates attention output＋recency features → hidden vector h.
- Point head (sigmoid) ⟹ pₜ = Pr(Xₜ = 1 | history)
- Count head (softmax) distribution over: Cₜ = ∑{i=0}^{K-1}X{t+i}

4. Loss = BCE(point) ＋ Cross-Entropy(count). The model learns a joint next-step and multi-step distribution.

5. Validation with time-series CV (forward expanding splits). Metrics:

- log-loss, Brier score, AUROC on the binary next-step task;
- Ranked Probability Score (RPS) and 95% coverage on the K-step count distribution.

6. Retraining on full series After CV, the best architecture is refit on the data and used to output:

- next-step probability,
- full distribution over the count in the next K steps,
- central-limit-theorem (CLT) baseline intervals for comparison.